# Soil Salinity Mapping & Crop Zoning — Fixed Build (v2: Punjab + Sindh only, real boundaries)

**Changes in this version:**

1. **Real province boundaries instead of rough rectangles.** Pulled from Earth Engine's `WM/geoLab/geoBoundaries` dataset (Pakistan ADM0 = country, ADM1 = provinces). Satellite extraction, the prediction grid, and the app's click-validation all now use the actual Punjab + Sindh polygons, not bounding boxes that leaked into the Arabian Sea / neighboring provinces / India.
2. **Map scope restricted to Pakistan.** The app draws a mask over everything outside Pakistan's border and locks pan/zoom to Pakistan's extent — other countries are no longer a usable part of the map.
3. **Punjab + Sindh are the only "active" provinces.** They're outlined/highlighted on the map. Clicking inside them returns a prediction; clicking elsewhere in Pakistan (Balochistan, KPK, etc.) returns a clear "no data for this region yet" message instead of a fake number.
4. **Click → marker + place name.** Clicking now drops a marker and reverse-geocodes the coordinate (via OpenStreetMap/Nominatim) to show a human-readable location name, alongside the predicted EC, salinity status, crop recommendation, and gypsum requirement.

5. **App was hanging on a blank screen.** The real province boundaries from `geoBoundaries` are very high-resolution (tens of thousands of coordinate points along Sindh's coastline). Rendering that much raw GeoJSON in Folium/Leaflet was hanging the page. Fixed: boundaries are now simplified (`shapely .simplify()`, ~1km tolerance) before being drawn — same shape, far fewer points, renders immediately.
6. **"Error: not connected to a server!"** — the `st_folium` map is a Streamlit custom component that opens its own websocket back to the server; cloudflared's default CORS/XSRF settings were blocking that handshake. Fixed: Streamlit is now launched with `--server.enableCORS false --server.enableXsrfProtection false`.

Carried over from the previous fix: real batched satellite extraction (`sampleRegions`), no synthetic-data override, no missing-CSV crash, no fake per-click numbers, cloudflared instead of localtunnel.

**Still true:** the EC labels used to train the model are placeholders (`np.random.uniform`), not real field measurements, for both Punjab and Sindh — I could not find a public, structured (lat/lon + EC) dataset for this region. Treat predictions as illustrative until real ground-truth is substituted in Cell 4.

## 1. Setup — Earth Engine

In [ ]:
# !pip install earthengine-api -q
import ee

PROJECT_ID = 'ee-salinity-project-502412'  # <-- your GCP project ID

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

print("Earth Engine initialized.")

## 2. Real province boundaries (FIXED — replaces the old 1°x1° rectangle bug)

In [ ]:
import json

adm0 = ee.FeatureCollection("WM/geoLab/geoBoundaries/600/ADM0")
adm1 = ee.FeatureCollection("WM/geoLab/geoBoundaries/600/ADM1")

pakistan_fc = adm0.filter(ee.Filter.eq('shapeGroup', 'PAK'))
pak_provinces_fc = adm1.filter(ee.Filter.eq('shapeGroup', 'PAK'))

# Sanity check: confirm the exact province name spellings in this dataset before filtering
province_names = pak_provinces_fc.aggregate_array('shapeName').getInfo()
print("Provinces found in dataset:", province_names)

In [ ]:
# Adjust these two strings if the printed list above spells them differently
PUNJAB_NAME = 'Punjab'
SINDH_NAME = 'Sindh'

target_provinces_fc = pak_provinces_fc.filter(
    ee.Filter.inList('shapeName', [PUNJAB_NAME, SINDH_NAME])
)
study_aoi = target_provinces_fc.geometry().dissolve(maxError=1)

# Pull geometries to the client and save as GeoJSON — the Streamlit app uses these
# for the "only Pakistan" mask, province outlines, and click-validation.
pakistan_geojson = pakistan_fc.geometry().getInfo()
target_provinces_geojson = target_provinces_fc.getInfo()

with open('pakistan_boundary.geojson', 'w') as f:
    json.dump({"type": "Feature", "geometry": pakistan_geojson, "properties": {}}, f)

with open('target_provinces.geojson', 'w') as f:
    json.dump(target_provinces_geojson, f)

print("Saved pakistan_boundary.geojson and target_provinces.geojson")

## 3. Sentinel-2 imagery + spectral indices, clipped to the real Punjab+Sindh polygon

In [ ]:
sentinel_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                       .filterBounds(study_aoi)
                       .filterDate('2023-01-01', '2023-12-31')
                       .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)))

clean_image = sentinel_collection.median().clip(study_aoi)

ndvi = clean_image.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndsi = clean_image.normalizedDifference(['B4', 'B8']).rename('NDSI')

final_image = clean_image.addBands([ndvi, ndsi]).select(['B2', 'B3', 'B4', 'B8', 'NDVI', 'NDSI'])
print("Bands ready:", final_image.bandNames().getInfo())

## 4. Ground-truth points — generated only *inside* the real province polygons

Still synthetic EC values (see note above). Coordinates are now rejection-sampled against the actual Punjab/Sindh shape instead of a rectangle, so no points fall in the sea or a neighboring country/province.

In [ ]:
import numpy as np
import pandas as pd
from shapely.geometry import shape, Point

with open('target_provinces.geojson') as f:
    target_gj = json.load(f)

province_shapes = [shape(feat['geometry']) for feat in target_gj['features']]
province_names_by_shape = [feat['properties'].get('shapeName', 'Unknown') for feat in target_gj['features']]
minx, miny, maxx, maxy = target_provinces_fc.geometry().bounds().getInfo()['coordinates'][0][0][0], \
                          target_provinces_fc.geometry().bounds().getInfo()['coordinates'][0][0][1], \
                          None, None
# simpler: compute bbox directly with shapely over all province shapes
from shapely.ops import unary_union
combined_shape = unary_union(province_shapes)
minx, miny, maxx, maxy = combined_shape.bounds

def sample_points_in_polygon(polygon, n, seed=42):
    rng = np.random.default_rng(seed)
    pts = []
    minx, miny, maxx, maxy = polygon.bounds
    while len(pts) < n:
        cand_lon = rng.uniform(minx, maxx, n)
        cand_lat = rng.uniform(miny, maxy, n)
        for lon, lat in zip(cand_lon, cand_lat):
            if polygon.contains(Point(lon, lat)):
                pts.append((lat, lon))
            if len(pts) >= n:
                break
    return pts

num_samples = 500
sampled_pts = sample_points_in_polygon(combined_shape, num_samples)

pk_ground_truth = pd.DataFrame(sampled_pts, columns=['Latitude', 'Longitude'])
pk_ground_truth['EC_GroundTruth'] = np.random.default_rng(42).uniform(0.5, 18.0, len(pk_ground_truth))  # PLACEHOLDER

print(f"{len(pk_ground_truth)} ground-truth points generated inside Punjab/Sindh polygon.")

## 5. Batch-extract real satellite features

In [ ]:
def df_to_feature_collection(df):
    feats = [ee.Feature(ee.Geometry.Point([row.Longitude, row.Latitude]), {'row_id': int(idx)})
             for idx, row in df.iterrows()]
    return ee.FeatureCollection(feats)

points_fc = df_to_feature_collection(pk_ground_truth)
sampled_fc = final_image.sampleRegions(collection=points_fc, scale=10, geometries=False)
sampled_records = sampled_fc.getInfo()['features']
sampled_df = pd.DataFrame([f['properties'] for f in sampled_records])

print(f"Extracted real satellite features for {len(sampled_df)} / {len(pk_ground_truth)} points.")

master_training_df = pk_ground_truth.merge(
    sampled_df.rename(columns={'row_id': 'row_index'}), left_index=True, right_on='row_index'
).drop(columns=['row_index'])

nir, red = master_training_df['B8'], master_training_df['B4']
master_training_df['DVI'] = nir - red
L = 0.5
master_training_df['SAVI'] = ((nir - red) / (nir + red + L)) * (1.0 + L)
master_training_df = master_training_df.dropna(subset=['NDVI', 'NDSI', 'DVI', 'SAVI', 'EC_GroundTruth'])
print(f"Final training table: {master_training_df.shape}")
master_training_df.head()

## 6. Train the model

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import joblib

features = ['NDVI', 'NDSI', 'DVI', 'SAVI']
X = master_training_df[features]
y = master_training_df['EC_GroundTruth']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_regressor = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42)
rf_regressor.fit(X_train, y_train)

y_pred = rf_regressor.predict(X_test)
print(f"R2:   {r2_score(y_test, y_pred):.4f}")
print(f"MAE:  {mean_absolute_error(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

plt.figure(figsize=(6, 5))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.7, edgecolor="w", s=60)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual EC (dS/m)'); plt.ylabel('Predicted EC (dS/m)')
plt.title('Random Forest: Predicted vs Actual Salinity')
plt.tight_layout(); plt.show()

joblib.dump(rf_regressor, 'pakistan_salinity_rf_model.pkl')
print("Saved: pakistan_salinity_rf_model.pkl")

## 7. Precompute the prediction grid — restricted to the real Punjab+Sindh polygon

In [ ]:
grid_step = 0.1  # degrees (~11km)

candidate_lats = np.arange(miny, maxy, grid_step)
candidate_lons = np.arange(minx, maxx, grid_step)
grid_pts = [(lat, lon) for lat in candidate_lats for lon in candidate_lons
            if combined_shape.contains(Point(lon, lat))]

grid_df = pd.DataFrame(grid_pts, columns=['Latitude', 'Longitude'])
print(f"Grid points inside Punjab/Sindh: {len(grid_df)}")

grid_fc = df_to_feature_collection(grid_df.assign(EC_GroundTruth=0))
grid_sampled = final_image.sampleRegions(collection=grid_fc, scale=10, geometries=False).getInfo()['features']
grid_sampled_df = pd.DataFrame([f['properties'] for f in grid_sampled])

grid_result = grid_df.merge(
    grid_sampled_df.rename(columns={'row_id': 'row_index'}), left_index=True, right_on='row_index'
).drop(columns=['row_index'])

nir, red = grid_result['B8'], grid_result['B4']
grid_result['DVI'] = nir - red
grid_result['SAVI'] = ((nir - red) / (nir + red + 0.5)) * 1.5
grid_result = grid_result.dropna(subset=features)
grid_result['Predicted_EC'] = rf_regressor.predict(grid_result[features])
grid_result[['Latitude', 'Longitude', 'Predicted_EC']].to_csv('salinity_grid.csv', index=False)

print(f"Saved salinity_grid.csv with {len(grid_result)} predicted points.")

## 8. Streamlit app — Pakistan-only view, Punjab+Sindh active, marker + place-name on click

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import json
import folium
from streamlit_folium import st_folium
from scipy.spatial import cKDTree
from shapely.geometry import shape, Point, box, mapping
from shapely.ops import unary_union
from geopy.geocoders import Nominatim

st.set_page_config(page_title="Pakistan Soil Tool", layout="wide")

@st.cache_resource
def load_assets():
    grid_df = pd.read_csv('salinity_grid.csv')
    tree = cKDTree(grid_df[['Latitude', 'Longitude']].values)

    SIMPLIFY_TOL = 0.01  # ~1km - keeps rendering fast without losing meaningful shape detail

    with open('pakistan_boundary.geojson') as f:
        pak_gj = json.load(f)
    pakistan_shape = shape(pak_gj['geometry']).simplify(SIMPLIFY_TOL, preserve_topology=True)

    with open('target_provinces.geojson') as f:
        raw_prov_gj = json.load(f)

    simplified_features = []
    province_shapes = {}
    for feat in raw_prov_gj['features']:
        name = feat['properties'].get('shapeName', 'Unknown')
        simp = shape(feat['geometry']).simplify(SIMPLIFY_TOL, preserve_topology=True)
        province_shapes[name] = simp
        simplified_features.append({
            "type": "Feature",
            "properties": {"shapeName": name},
            "geometry": mapping(simp)
        })
    prov_gj = {"type": "FeatureCollection", "features": simplified_features}

    active_shape = unary_union(list(province_shapes.values()))
    world = box(-180, -90, 180, 90)
    mask_shape = world.difference(pakistan_shape)

    return grid_df, tree, pakistan_shape, prov_gj, active_shape, mask_shape

def get_recommendation(ec_value):
    if ec_value < 4.0:
        return {"status": "Normal / Slight Salinity", "crops": ["wheat", "cotton", "rice"]}
    elif ec_value < 8.0:
        return {"status": "Moderate Salinity", "crops": ["sunflower", "sugarcane"]}
    else:
        return {"status": "Severe Salinity", "crops": ["fodder crops", "olive", "quinoa", "Salicornia"]}

def calculate_gypsum(ec_value):
    if ec_value <= 4.0:
        return 0.0, 0.0
    tons = (ec_value - 4.0) * 0.4
    return round(tons, 2), round(tons * 22500, 2)

@st.cache_data(show_spinner=False)
def reverse_geocode(lat, lon):
    try:
        geolocator = Nominatim(user_agent="pak_salinity_mapper")
        location = geolocator.reverse(f"{lat}, {lon}", zoom=10, timeout=5)
        return location.address if location else "Unknown location"
    except Exception:
        return "Location name unavailable"

def main():
    if 'target_lat' not in st.session_state:
        st.session_state.target_lat = None
        st.session_state.target_lon = None

    try:
        grid_df, tree, pakistan_shape, prov_gj, active_shape, mask_shape = load_assets()
    except Exception as e:
        st.error(f"Could not load required files: {e}")
        return

    st.title("Pakistan Spatial Soil Mapping")
    st.caption("Click anywhere in Punjab or Sindh for a salinity prediction and crop guidance. "
               "(Balochistan / KPK are shown for context but have no data yet.)")
    col1, col2 = st.columns([1.5, 1.2])

    with col1:
        pak_minx, pak_miny, pak_maxx, pak_maxy = pakistan_shape.bounds
        m = folium.Map(
            location=[30.0, 69.5], zoom_start=6, tiles="CartoDB positron",
            max_bounds=True,
            min_lat=pak_miny - 1, max_lat=pak_maxy + 1,
            min_lon=pak_minx - 1, max_lon=pak_maxx + 1,
            max_bounds_viscosity=1.0
        )
        m.fit_bounds([[pak_miny, pak_minx], [pak_maxy, pak_maxx]])

        # Grey out everything outside Pakistan
        folium.GeoJson(
            mapping(mask_shape),
            style_function=lambda f: {"fillColor": "#1a1a1a", "color": "#1a1a1a",
                                       "fillOpacity": 0.55, "weight": 0},
            interactive=False
        ).add_to(m)

        # Pakistan outline
        folium.GeoJson(
            mapping(pakistan_shape),
            style_function=lambda f: {"fillOpacity": 0, "color": "#333333", "weight": 1.5},
            interactive=False
        ).add_to(m)

        # Punjab / Sindh highlighted as the active data provinces
        folium.GeoJson(
            prov_gj,
            style_function=lambda f: {"fillColor": "#2e7d32", "color": "#1b5e20",
                                       "weight": 2, "fillOpacity": 0.12},
            tooltip=folium.GeoJsonTooltip(fields=["shapeName"], aliases=["Province:"]),
            interactive=False
        ).add_to(m)

        if st.session_state.target_lat:
            folium.Marker(
                [st.session_state.target_lat, st.session_state.target_lon],
                icon=folium.Icon(color="darkblue")
            ).add_to(m)

        map_data = st_folium(m, height=550, width=650)
        if map_data.get('last_clicked'):
            new_lat, new_lon = map_data['last_clicked']['lat'], map_data['last_clicked']['lng']
            if (new_lat, new_lon) != (st.session_state.target_lat, st.session_state.target_lon):
                st.session_state.target_lat, st.session_state.target_lon = new_lat, new_lon
                st.rerun()

    with col2:
        lat, lon = st.session_state.target_lat, st.session_state.target_lon
        if lat is None:
            st.info("Click a point on the map to get a prediction.")
        elif not pakistan_shape.contains(Point(lon, lat)):
            st.warning("That point is outside Pakistan. Click within the country border.")
        elif not active_shape.contains(Point(lon, lat)):
            st.warning("No data available for this region yet — only Punjab and Sindh are covered so far.")
            place_name = reverse_geocode(lat, lon)
            st.markdown(f"**Location:** {place_name}")
        else:
            place_name = reverse_geocode(lat, lon)
            dist, idx = tree.query([lat, lon])
            row = grid_df.iloc[idx]
            predicted_ec = row['Predicted_EC']
            rec = get_recommendation(predicted_ec)

            st.markdown(f"### Location: {place_name}")
            st.markdown(f"**Coordinates:** {lat:.4f}, {lon:.4f}")
            st.markdown(f"### Predicted Salinity: {predicted_ec:.2f} dS/m")
            st.markdown(f"### Status: {rec['status']}")
            st.markdown(f"**Recommended crops:** {', '.join(rec['crops'])}")

            tons, pkr_cost = calculate_gypsum(predicted_ec)
            if tons > 0:
                st.error(f"Gypsum Required: {tons} Tons/Acre (Rs. {pkr_cost:,.0f})")

            report_data = pd.DataFrame([{
                "Location": place_name, "Lat": lat, "Lon": lon,
                "Predicted_EC": predicted_ec, "Status": rec['status']
            }])
            st.download_button("Download Report", report_data.to_csv(index=False), "report.csv", "text/csv")

if __name__ == "__main__":
    main()

## 9. Launch

In [ ]:
!pip install streamlit streamlit-folium folium scipy shapely geopy -q
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

!pkill -f streamlit || true
!pkill -f cloudflared || true
!pkill -f localtunnel || true
print("Cleaned up any previous Streamlit/tunnel processes.")

In [ ]:
import subprocess, time

# --server.enableCORS false / --server.enableXsrfProtection false:
# without these, the st_folium map component's websocket handshake gets blocked
# when served through the cloudflared tunnel ("Error: not connected to a server!").
subprocess.Popen(
    "streamlit run app.py --server.port 8501 --server.headless true "
    "--server.enableCORS false --server.enableXsrfProtection false "
    "> /content/streamlit_logs.txt 2>&1",
    shell=True
)
time.sleep(6)
subprocess.Popen("./cloudflared tunnel --url http://localhost:8501 > /content/cloudflared_logs.txt 2>&1", shell=True)
time.sleep(8)

!grep -o 'https://[a-zA-Z0-9.-]*trycloudflare.com' /content/cloudflared_logs.txt | head -1